In [86]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import column
import os
path = kagglehub.dataset_download("osmi/mental-health-in-tech-2016")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\ugniu\.cache\kagglehub\datasets\osmi\mental-health-in-tech-2016\versions\2


In [87]:
# creating a separate  file for data manipulation
df_raw = pd.read_csv( r"C:\Users\ugniu\PycharmProjects\Unsupervised  learning\Data\Raw\mental-heath-in-tech-2016_20161114.csv")

df = df_raw.copy()


In [88]:
# The first step: Removing columns that contain answers to open questions(are not  generalizible) and Encoding the values so we can perform K-Clustering analysis.

#Row 1 "Are you self employed" - Binary, no correction needed, looks good

#Row 2 "How many employees does your company or organization have?"   - encoding the row, creating separate binary columns.

df =pd.get_dummies(df, columns= ["How many employees does your company or organization have?"],  prefix="company_size",dummy_na=True)

#Row 3 Is your employer primarily a tech company/organization? - 287 NA values matches the 287 Self-employed people. Based on this  we  can assume, this number represent this group of people. However, we are not entirely sure, therefore we will keep the NA values.


df = pd.get_dummies(df, columns=["Is your employer primarily a tech company/organization?"],  prefix= "is_tech_comp?", dummy_na= True)



#Row 4 -Is your primary role within your company related to tech/IT? -  NA values are overwhelmingly high, 83%, therefore we will drop ths column

df = df.drop(columns = "Is your primary role within your company related to tech/IT?")



#Row 5 "Does your employer provide mental health benefits as part of healthcare coverage?" - since this is not an ordinal column, we will create dummy variables. Additionally, NA values matches the self-employed numbers (287),but there is no guarantee that this number represents exactly that. 287 NA will repeatedly show up in NA values wth this dataset.


df = pd.get_dummies(df, columns=["Does your employer provide mental health benefits as part of healthcare coverage?"], prefix="healthcare_benefits", dummy_na= True)

#Row 6 "Do you know the options for mental health care available under your employer-provided coverage?"  Not an ordinal column  - create dummy variables. The NA values are significantly higher than the self-employed number.


df =  pd.get_dummies(df, columns=["Do you know the options for mental health care available under your employer-provided coverage?"], prefix = "healthcare_awareness",dummy_na= True)

# Row 7 "Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?"  - NA column matches the number of self-employed (287).


df =  pd.get_dummies(df, columns=["Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?"], prefix = "healthcare_employer_discussion",dummy_na=True)

#Row 8 Does your employer offer resources to learn more about mental health concerns and options for seeking help?

df =  pd.get_dummies(df, columns=["Does your employer offer resources to learn more about mental health concerns and options for seeking help?"], prefix = "healthcare_employer_resources",dummy_na=True)

#Row 9 Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?

df =  pd.get_dummies(df, columns=["Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?"], prefix = "healthcare_anonymity",dummy_na=True)

#  Row 10 If a mental health issue prompted you to request a medical leave from work, asking for that leave would be: , ths is an ordinal column, and we can assign a rating structure. Additionally,  NAN - might represent self-employed 287. Creating two new columns - one for ordinal rating and the second for NA and "I dont know" values in binary format. We will drop the original column in the process.

order_row_10 ={"Very difficult": 1,"Somewhat difficult": 2,"Neither easy nor difficult": 3,"Somewhat easy": 4,"Very easy": 5}

df["medical_leave_ordinal"] = df["If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:"].map(order_row_10)

df["medical_leave_dont_know"] = (df["If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:"] == "I don't know").astype(int)

df = df.drop(columns = ["If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:"])




In [89]:
#Row 11 Do you think that discussing a mental health disorder with your employer would have negative consequences? - answers - yes, no, maybe. This has potentially ordinal structure, therefore, we will assign 1 to no, 2 to maybe and 3  to yes. We also drop the column

order_column_maybe= {"No": 1, "Maybe": 2  , "Yes": 3}

df["discussing_mental_employer_ordinal"] = df["Do you think that discussing a mental health disorder with your employer would have negative consequences?"].map(order_column_maybe)

df = df.drop(columns =["Do you think that discussing a mental health disorder with your employer would have negative consequences?"])

# Row 12 Do you think that discussing a physical health issue with your employer would have negative consequences?
# Same encoding as the last  column

df["discussing_physical_employer_ordinal"] = df["Do you think that discussing a physical health issue with your employer would have negative consequences?"].map(order_column_maybe)

df = df.drop(columns="Do you think that discussing a physical health issue with your employer would have negative consequences?")

#Row 13 Would you feel comfortable discussing a mental health disorder with your coworkers?
#NA = 287
# has the same structure  - no,yes,maybe

df["discussing_mental_coworkers_ordinal"] = df["Would you feel comfortable discussing a mental health disorder with your coworkers?"].map(order_column_maybe)

df = df.drop(columns="Would you feel comfortable discussing a mental health disorder with your coworkers?")


#Row 14 Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?
#same  structure

df["discussing_mental_supervisor_ordinal"] = df["Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?"].map(order_column_maybe)

df = df.drop(columns="Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?")

#Row 15 Do you feel that your employer takes mental health as seriously as physical health?
# Has - yes, no,  I dont know. We will create dummy variables

df =  pd.get_dummies(df, columns=["Do you feel that your employer takes mental health as seriously as physical health?"], prefix = "employer_take_health",dummy_na=True)

#Row 16 Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?
# contains yes and no data. conversion to binary needed

order_column_17 = {"No":0,"Yes":1}

df["Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?"] = df["Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?"].map(order_column_17)

#col 17 Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?
#Contains over 80% NA values (1146 total), therefore will be dropped

df=  df.drop(columns= "Do you have medical coverage (private insurance or state-provided) which includes treatment of  mental health issues?")

#col 18 Do you know local or online resources to seek help for a mental health disorder?
# also contains 1146 NA values = drop

df=  df.drop(columns= "Do you know local or online resources to seek help for a mental health disorder?")

# Col 19 Do you know local or online resources to seek help for a mental health disorder?
# 1146 NA values =  drop

df=  df.drop(columns= "If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?")

#Col 20 - If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?

# 1289 NA =  drop

df=  df.drop(columns= "If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?")

#Col 21 -If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?

df=  df.drop(columns= "If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?")

#Col 22 - If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?
# 1146  NA values = drop

df=  df.drop(columns= "If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?")

#Col 23 - Do you believe your productivity is ever affected by a mental health issue?
#1146 NA values= drop

df=  df.drop(columns= "Do you believe your productivity is ever affected by a mental health issue?")

#Col 24  -  If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?
#1229 NA

df=  df.drop(columns= "If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?")

#Col 25 Do you have previous employers?
#Binary  column -all good


In [90]:
#Col 26 - Have your previous employers provided mental health benefits?
# There is an ordinal structure - No, some did, yes.
#"I dont know " does not provide any useful info, therefore conversion to NA

order_column_26 = {"No, none did": 0 , "Some did" : 1 , "Yes, they all did":2, "I don't know": np.nan}

df["Have your previous employers provided mental health benefits?"] = df["Have your previous employers provided mental health benefits?"].map(order_column_26)


#Col 27 - Were you aware of the options for mental health care provided by your previous employers?
# Ordinal column structure - NA, "No, I only became aware later", "I was aware of some","Yes, I was aware of all of them"

order_column_27= {"N/A (not currently aware)":0, "No, I only became aware later":1, "I was aware of some":2,"Yes, I was aware of all of them":3}

df["Were you aware of the options for mental health care provided by your previous employers?"] = df["Were you aware of the options for mental health care provided by your previous employers?"].map(order_column_27)


#Col 28 -Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?

#Ordinal column structure

order_column_28  = {"I don't know": np.nan, "None did": 0, "Some did":1, "Yes, they all did":2}

df["Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?"] = df["Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?"].map(order_column_28)

#Col 29 - Did your previous employers provide resources to learn more about mental health issues and how to seek help?

#Ordinary column
#Same answer categories as 28th column

order_column_29 = order_column_28

df["Did your previous employers provide resources to learn more about mental health issues and how to seek help?"] = df["Did your previous employers provide resources to learn more about mental health issues and how to seek help?"].map(order_column_29)

#Col 30 - Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?

order_column_30  = {"I don't know": np.nan, "No": 0, "Sometimes":1, "Yes, always":2}

df["Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?"] = df["Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?"].map(order_column_30)



In [91]:
#Col 31 - Do you think that discussing a mental health disorder with previous employers would have negative consequences?

#Ordindal column

order_column_31  = {"I don't know": np.nan, "None of them": 0, "Some of them":1, "Yes, all of them":2}

df["Do you think that discussing a mental health disorder with previous employers would have negative consequences?"] = df["Do you think that discussing a mental health disorder with previous employers would have negative consequences?"].map(order_column_31)


#Col 32 - Do you think that discussing a physical health issue with previous employers would have negative consequences?

#Order column

order_column_32  = { "None of them": 0, "Some of them":1, "Yes, all of them":2}

df["Do you think that discussing a physical health issue with previous employers would have negative consequences?"]= df["Do you think that discussing a physical health issue with previous employers would have negative consequences?"].map(order_column_32)


# Col 33 - Would you have been willing to discuss a mental health issue with your previous co-workers?

order_column_33 = {"No, at none of my previous employers": 0, "Some of my previous employers": 1, "Yes, at all of my previous employers": 2}

df["Would you have been willing to discuss a mental health issue with your previous co-workers?"] = df["Would you have been willing to discuss a mental health issue with your previous co-workers?"].map(order_column_33)


#Col 34 - Would you have been willing to discuss a mental health issue with your direct supervisor(s)?

order_column_34  = {"I don't know": np.nan, "No, at none of my previous employers": 0, "Some of my previous employers":1, "Yes, at all of my previous employers":2}

df["Would you have been willing to discuss a mental health issue with your direct supervisor(s)?"] = df["Would you have been willing to discuss a mental health issue with your direct supervisor(s)?"].map(order_column_34)



#Col 35  - Did you feel that your previous employers took mental health as seriously as physical health?

order_column_35  = {"I don't know": np.nan, "None did": 0, "Some did":1, "Yes, they all did":2}

df["Did you feel that your previous employers took mental health as seriously as physical health?"]  = df["Did you feel that your previous employers took mental health as seriously as physical health?"].map(order_column_35)


#Col 36 - Did you hear of or observe negative consequences for co-workers with mental health issues in your previous workplaces?

order_column_36  = {"I don't know": np.nan, "None of them": 0, "Some of them":1, "Yes, all of them":2}

df["Did you hear of or observe negative consequences for co-workers with mental health issues in your previous workplaces?"] = df["Did you hear of or observe negative consequences for co-workers with mental health issues in your previous workplaces?"].map(order_column_36)


#Col 37 - Would you be willing to bring up a physical health issue with a potential employer in an interview?

#No,Maybe, Yes

order_column_37= {"No":0, "Maybe":1,"Yes":2}

df["Would you be willing to bring up a physical health issue with a potential employer in an interview?"] = df["Would you be willing to bring up a physical health issue with a potential employer in an interview?"].map(order_column_37)

# Col 38 - Why or why not?

#Open text, not generalizable, therefore drop

df = df.drop(columns=["Why or why not?"])


# Col 39 Would you bring up a mental health issue with a potential employer in an interview?

order_column_39= {"No":0, "Maybe":1,"Yes":2}

df["Would you bring up a mental health issue with a potential employer in an interview?"] = df["Would you bring up a mental health issue with a potential employer in an interview?"].map(order_column_39)

#Col 40- Why or why not?.1

#Also open question, drop.
df = df.drop(columns=["Why or why not?.1"])


In [92]:
# Col 41- Do you feel that being identified as a person with a mental health issue would hurt your career?

#Ordinal column

order_column_41 = {"No, it has not":  0,  "No, I don't think it would" :1 , "Maybe":2,"Yes, I think it would":3, "Yes, it has":4}

df["Do you feel that being identified as a person with a mental health issue would hurt your career?"] = df["Do you feel that being identified as a person with a mental health issue would hurt your career?"].map(order_column_41)


#Col 42 - Do you think that team members/co-workers would view you more negatively if they knew you suffered from a mental health issue?

order_column_42 = {"No, they do not":  0,  "No, I don't think they would" :1 , "Maybe":2,"Yes, I think they would":3, "Yes, they do":4}

df["Do you think that team members/co-workers would view you more negatively if they knew you suffered from a mental health issue?"] = df["Do you think that team members/co-workers would view you more negatively if they knew you suffered from a mental health issue?"].map(order_column_42)

#Col 43 How willing would you be to share with friends and family that you have a mental illness?

order_column_43 = {"Not applicable to me (I do not have a mental illness)":  np.nan,"Not open at all":  0,  "Somewhat not open" :1 , "Neutral":2,"Somewhat open":3, "Very open":4}

df["How willing would you be to share with friends and family that you have a mental illness?"] = df["How willing would you be to share with friends and family that you have a mental illness?"].map(order_column_43)

#Col 44 - Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?

#Ordinal column

order_column_44 = {"No":  0,  "Maybe/Not sure" :1 , "Yes, I experienced":2,"SYes, I observed":3}

df["Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?"]= df["Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?"].map(order_column_44)


#Col 45 Have your observations of how another individual who discussed a mental health disorder made you less likely to reveal a mental health issue yourself in your current workplace?

#Very high NA count , therefore drop

df= df.drop(columns= ["Have your observations of how another individual who discussed a mental health disorder made you less likely to reveal a mental health issue yourself in your current workplace?"])

#Col 46 Do you have a family history of mental illness?
#Col 46 Do you have a family history of mental illness?
# A conversion to binary column

order_column_46= {"No": 0, "Yes" :1, "I don't know":  np.nan}

df["Do you have a family history of mental illness?"] = df["Do you have a family history of mental illness?"].map(order_column_46)

#Col 47 Have you had a mental health disorder in the past?
#ordinal column


order_column_47= {"No": 0, "Yes" :2, "Maybe": 1 }

df["Have you had a mental health disorder in the past?"] = df["Have you had a mental health disorder in the past?"].map(order_column_47)

#Col 48 Do you currently have a mental health disorder?

#ordinal

order_column_48= {"No": 0, "Yes" :2, "Maybe": 1 }

df["Do you currently have a mental health disorder?"] = df["Do you currently have a mental health disorder?"].map(order_column_48)

#Col 49 If yes, what condition(s) have you been diagnosed with?
# This question is an extension for the 575 people that have answered  yes in the 48 question

# there are 117 unique answers,  but they seem to be uncategorized, therefore, the values will be standardized
# Addiionally, we will split the values by || and do dummy  variables


# first we create new  columns by splitting the the answer categories
disorder_dummies = df["If yes, what condition(s) have you been diagnosed with?"].str.get_dummies(sep="|")


# Now we standardize the answer values
condition_map = {
    # ADHD
    "condition_ADD (w/o Hyperactivity)": "ADHD",
    "condition_ADD (w/o Hyperactivity).1": "ADHD",
    "condition_Attention Deficit Hyperactivity Disorder": "ADHD",
    "condition_Attention Deficit Hyperactivity Disorder.1": "ADHD",
    "condition_Attention Deficit Hyperactivity Disorder.2": "ADHD",
    "condition_Attention Deficit Disorder": "ADHD",
    "condition_attention deficit disorder (but not the hyperactive version)": "ADHD",
    'condition_MCD (when it was diagnosed, the ultra-mega "disorder" ADHD didn\'t exist yet)': "ADHD",
    'condition_Combination of physical impairment (strongly near-sighted) with a possibly mental one (MCD / "ADHD", though its actually a stimulus filtering impairment)': "ADHD",

    # Autism
    "condition_Autism": "Autism", "condition_Autism.1": "Autism", "condition_Autism.2": "Autism",
    "condition_Autism (Asperger's)": "Autism", "condition_Autism (Asperger's).1": "Autism",
    "condition_Autism Spectrum Disorder": "Autism", "condition_Autism Spectrum Disorder.1": "Autism",
    "condition_Autism spectrum disorder": "Autism", "condition_autism spectrum disorder": "Autism",
    "condition_Asperges": "Autism", "condition_Asperger Syndrome": "Autism",
    "condition_Asperger Syndrome.1": "Autism", "condition_Asperger's": "Autism", "condition_Aspergers": "Autism",
    "condition_PDD-NOS": "Autism", "condition_PDD-NOS.1": "Autism", "condition_PDD-NOS (see above)": "Autism",
    "condition_Pervasive Developmental Disorder (Not Otherwise Specified)": "Autism",
    "condition_Autism - while not a \"mental illness\", still greatly affects how I handle anxiety": "Autism",

    # Mood Disorder
    "condition_Mood Disorder (Depression, Bipolar Disorder, etc)": "Mood Disorder",
    "condition_Mood Disorder (Depression, Bipolar Disorder, etc).1": "Mood Disorder",
    "condition_Mood Disorder (Depression, Bipolar Disorder, etc).2": "Mood Disorder",
    "condition_Depression": "Mood Disorder", "condition_Depression.1": "Mood Disorder",
    "condition_Seasonal Affective Disorder": "Mood Disorder", "condition_Seasonal Affective Disorder.1": "Mood Disorder",
    "condition_I haven't been formally diagnosed, so I felt uncomfortable answering, but Social Anxiety and Depression.": "Mood Disorder",
    "condition_Intimate Disorder":  "Mood Disorder",

    # Anxiety
    "condition_Anxiety Disorder (Generalized, Social, Phobia, etc)": "Anxiety",
    "condition_Anxiety Disorder (Generalized, Social, Phobia, etc).1": "Anxiety",
    "condition_Anxiety Disorder (Generalized, Social, Phobia, etc).2": "Anxiety",
    "condition_post-partum / anxiety": "Anxiety",

    # PTSD
    "condition_PTSD (undiagnosed)": "PTSD",
    "condition_Post-traumatic Stress Disorder": "PTSD", "condition_Post-traumatic Stress Disorder.1": "PTSD",
    "condition_Post-traumatic Stress Disorder.2": "PTSD", "condition_posttraumatic stress disourder": "PTSD",

    # OCD
    "condition_Obsessive-Compulsive Disorder": "OCD",
    "condition_Obsessive-Compulsive Disorder.1": "OCD", "condition_Obsessive-Compulsive Disorder.2": "OCD",

    # Eating Disorder
    "condition_Eating Disorder (Anorexia, Bulimia, etc)": "Eating Disorder",
    "condition_Eating Disorder (Anorexia, Bulimia, etc).1": "Eating Disorder",
    "condition_Eating Disorder (Anorexia, Bulimia, etc).2": "Eating Disorder",

    # Substance/Addictive
    "condition_Addictive Disorder": "Substance/Addictive Disorder",
    "condition_Addictive Disorder.1": "Substance/Addictive Disorder", "condition_Addictive Disorder.2": "Substance/Addictive Disorder",
    "condition_Substance Use Disorder": "Substance/Addictive Disorder",
    "condition_Substance Use Disorder.1": "Substance/Addictive Disorder", "condition_Substance Use Disorder.2": "Substance/Addictive Disorder",
     "condition_Sexual addiction": "Substance/Addictive Disorder",

    # Dissociative Disorder
    "condition_Dissociative Disorder": "Dissociative Disorder",
    "condition_Dissociative Disorder.1": "Dissociative Disorder", "condition_Dissociative Disorder.2": "Dissociative Disorder",
    "condition_Depersonalisation": "Dissociative Disorder", "condition_depersonalization disorder": "Dissociative Disorder",

    # Personality Disorder (incl. Schizotypal as a subtype)
    "condition_Personality Disorder (Borderline, Antisocial, Paranoid, etc)": "Personality Disorder",
    "condition_Personality Disorder (Borderline, Antisocial, Paranoid, etc).1": "Personality Disorder",
    "condition_Personality Disorder (Borderline, Antisocial, Paranoid, etc).2": "Personality Disorder",
    "condition_Schizotypal Personality Disorder": "Personality Disorder",
    "condition_Schizotypal Personality Disorder.1": "Personality Disorder",

    # Psychotic Disorder
    "condition_Psychotic Disorder (Schizophrenia, Schizoaffective, etc)": "Psychotic Disorder",
    "condition_Psychotic Disorder (Schizophrenia, Schizoaffective, etc).1": "Psychotic Disorder",
    "condition_Psychotic Disorder (Schizophrenia, Schizoaffective, etc).2": "Psychotic Disorder",

    # Stress Response Syndromes
    "condition_Stress Response Syndromes": "Stress Response Syndromes",
    "condition_Stress Response Syndromes.1": "Stress Response Syndromes", "condition_Stress Response Syndromes.2": "Stress Response Syndromes",

    # Gender Dysphoria
    "condition_Gender Dysphoria": "Gender Dysphoria", "condition_Gender Dysphoria.1": "Gender Dysphoria",
    "condition_Gender Identity Disorder": "Gender Dysphoria", "condition_Gender Identity Disorder.1": "Gender Dysphoria",

    # Suicidal Ideation
    "condition_Suicidal Ideation": "Suicidal Ideation", "condition_Suicidal Ideation.1": "Suicidal Ideation",

    # Single-occurrence, no variants
    "condition_Sleeping Disorder": "Sleeping Disorder",
    "condition_Traumatic Brain Injury": "Traumatic Brain Injury",
    "condition_Burn out": "Burnout", "condition_Burn out.1": "Burnout", "condition_Burnout": "Burnout",
}

condition_map_clean = {
    key.replace("condition_", "", 1): value
    for key, value in condition_map.items()
}



# Standardize condition names
disorder_dummies_yes = disorder_dummies.rename(columns=condition_map_clean)


# Combine duplicate columns that map to the same standardized condition
disorder_dummies_yes = disorder_dummies_yes.T.groupby(level=0).max().T


# Add prefix to standardized columns
disorder_dummies_yes = disorder_dummies_yes.add_prefix("condition_")


# Add the new variables to df
df = pd.concat([df, disorder_dummies_yes], axis=1)


# Drop the original condition column and the "transgender" column
df = df.drop(
    columns=["If yes, what condition(s) have you been diagnosed with?","condition_Transgender"]
)




#50  If maybe, what condition(s) do you believe you have?
#same logic here
#Due to high correlation with previously stated question, we will drop this columns


df = df.drop(columns=["If maybe, what condition(s) do you believe you have?"])

#Col 51 Have you been diagnosed with a mental health condition by a medical professional?
# simple binary conversion

order_column_51 = {"Yes": 1, "No":0}

df["Have you been diagnosed with a mental health condition by a medical professional?"]= df["Have you been diagnosed with a mental health condition by a medical professional?"].map(order_column_51)

#Col 52 If so, what condition(s) were you diagnosed with?
#Also dropping this one, because it will correlate with previous questions.

df = df.drop(columns=["If so, what condition(s) were you diagnosed with?"])


#Col 53 Have you ever sought treatment for a mental health issue from a mental health professional?
#binary = good

#Col 54 If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?
#ordinal column

order_column_54 = {"Not applicable to me": np.nan, "Never":0, "Rarely":1, "Sometimes":2, "Often":3}

df["If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?"] = df["If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?"].map(order_column_54)




In [93]:
#Col 55 If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?

order_column_55 = {"Not applicable to me": np.nan, "Never":0, "Rarely":1, "Sometimes":2, "Often":3}

df["If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?"] = df["If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?"].map(order_column_55)

#Col 56 What is your age?

#there are some outliers and rows need sanity  check

df = df[df["What is your age?"].between(18, 80)]

#Col 57 What is your gender?

# many type errors

#creating 3  groups - Male, Female, the rest is non-binary

#Cisgender = birth gender

col = "What is your gender?"

gender_map = {
    # Male
    "Male": "Male", "male": "Male", "M": "Male", "m": "Male", "Male ": "Male", "male ": "Male",
    "Male (cis)": "Male", "Man": "Male", "man": "Male", "Cis male": "Male", "cis male": "Male",
    "Male.": "Male", "Sex is male": "Male", "cis man": "Male", "Malr": "Male", "mail": "Male",
    "M|": "Male", "Cis Male": "Male", "cisdude": "Male", "Dude": "Male", "MALE": "Male",
    "I'm a man why didn't you make this a drop down question. You should of asked sex? And I would of answered yes please. Seriously how much text can this take? ": "Male",

    # Female
    "Female": "Female", "female": "Female", "F": "Female", "f": "Female", "Female ": "Female",
    "female ": "Female", "woman": "Female", "Woman": "Female", "I identify as female.": "Female",
    "fm": "Female", "Cis female ": "Female", "female/woman": "Female", "Cisgender Female": "Female",
    " Female": "Female", "Female (props for making this a freeform field, though)": "Female",
    "fem": "Female", "Cis-woman": "Female",

    # Non-binary / trans / genderqueer / other explicit non-binary identities
    "non-binary": "Non-binary", "Nonbinary": "Non-binary", "Agender": "Non-binary",
    "Bigender": "Non-binary", "Female assigned at birth ": "Non-binary",
    "Transitioned, M2F": "Non-binary", "Genderfluid (born female)": "Non-binary",
    "Other/Transfeminine": "Non-binary", "Female or Multi-Gender Femme": "Non-binary",
    "Androgynous": "Non-binary", "Other": "Non-binary", "nb masculine": "Non-binary",
    "genderqueer": "Non-binary", "Genderfluid": "Non-binary", "Enby": "Non-binary",
    "genderqueer woman": "Non-binary", "mtf": "Non-binary", "Queer": "Non-binary",
    "Fluid": "Non-binary", "Male/genderqueer": "Non-binary", "Genderqueer": "Non-binary",
    "Genderflux demi-girl": "Non-binary", "AFAB": "Non-binary", "Transgender woman": "Non-binary",
    "Male (trans, FtM)": "Non-binary", "male 9:1 female, roughly": "Non-binary",

    # genuinely unclear / not a real gender answer -> NA
    "none of your business": "NA", "Human": "NA", "human": "NA", "Unicorn": "NA",
    "female-bodied; no feelings about gender": "NA",
}

df["gender_clean"] = df[col].map(gender_map)
df["gender_clean"] = df["gender_clean"].fillna("NA")  # catches the 3 true NaN values too

df = pd.get_dummies(df, columns=["gender_clean"], prefix="gender")
df = df.drop(columns=[col])

#Geographical data manipulation

#```````````````````````````````````````````````````````````````````````````````````````````````````````````````````
state_to_region = {
    # Northeast
    "Connecticut": "Northeast", "Maine": "Northeast", "Massachusetts": "Northeast",
    "New Hampshire": "Northeast", "Rhode Island": "Northeast", "Vermont": "Northeast",
    "New Jersey": "Northeast", "New York": "Northeast", "Pennsylvania": "Northeast",
    # Midwest
    "Illinois": "Midwest", "Indiana": "Midwest", "Iowa": "Midwest", "Kansas": "Midwest",
    "Michigan": "Midwest", "Minnesota": "Midwest", "Missouri": "Midwest", "Nebraska": "Midwest",
    "North Dakota": "Midwest", "Ohio": "Midwest", "South Dakota": "Midwest", "Wisconsin": "Midwest",
    # Southwest
    "Arizona": "Southwest", "New Mexico": "Southwest", "Oklahoma": "Southwest", "Texas": "Southwest",
    # Southeast
    "Alabama": "Southeast", "Delaware": "Southeast", "District of Columbia": "Southeast",
    "Florida": "Southeast", "Georgia": "Southeast", "Kentucky": "Southeast", "Louisiana": "Southeast",
    "Maryland": "Southeast", "North Carolina": "Southeast", "South Carolina": "Southeast",
    "Tennessee": "Southeast", "Virginia": "Southeast", "West Virginia": "Southeast",
    # West
    "Alaska": "West", "California": "West", "Colorado": "West", "Idaho": "West",
    "Montana": "West", "Nevada": "West", "Oregon": "West", "Utah": "West", "Washington": "West",
}

# Country -> continent
country_to_continent = {
    "Algeria": "Africa", "South Africa": "Africa",
    "Afghanistan": "Asia", "Bangladesh": "Asia", "Brunei": "Asia", "China": "Asia",
    "India": "Asia", "Iran": "Asia", "Israel": "Asia", "Japan": "Asia",
    "Pakistan": "Asia", "Taiwan": "Asia", "Vietnam": "Asia",
    "Austria": "Europe", "Belgium": "Europe", "Bosnia and Herzegovina": "Europe",
    "Bulgaria": "Europe", "Czech Republic": "Europe", "Denmark": "Europe", "Estonia": "Europe",
    "Finland": "Europe", "France": "Europe", "Germany": "Europe", "Greece": "Europe",
    "Hungary": "Europe", "Ireland": "Europe", "Italy": "Europe", "Lithuania": "Europe",
    "Netherlands": "Europe", "Norway": "Europe", "Poland": "Europe", "Romania": "Europe",
    "Russia": "Europe", "Serbia": "Europe", "Slovakia": "Europe", "Spain": "Europe",
    "Sweden": "Europe", "Switzerland": "Europe", "United Kingdom": "Europe",
    "Canada": "North America", "United States of America": "North America",
    "Mexico": "North America", "Costa Rica": "North America", "Guatemala": "North America",
    "Argentina": "South America", "Brazil": "South America", "Chile": "South America",
    "Colombia": "South America", "Ecuador": "South America", "Venezuela": "South America",
    "Australia": "Oceania", "New Zealand": "Oceania",
    "Other": "Other/Unspecified",
}




#Col 58 - What country do you live in?

#Nominal data = dummy variables



df["continent"] = df["What country do you live in?"].map(country_to_continent).fillna("Na")
df = pd.get_dummies(df, columns=["continent"], prefix="continent", dummy_na=True)
df = df.drop(columns=["What country do you live in?"])



#Col 59 What US state or territory do you live in?
#Nominal data = dummies

df["us_region"] = df["What US state or territory do you live in?"].map(state_to_region).fillna("NA")
df = pd.get_dummies(df, columns=["us_region"], prefix="us_region")
df = df.drop(columns=["What US state or territory do you live in?"])



# These columns will be dropped, because of high correlation with residency location
#Col 60 What country do you work in?
#Col 61 - What US state or territory do you work in?

df = df.drop(columns = ["What country do you work in?",  "What US state or territory do you work in?"])






#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

#Col 62  - Which of the following best describes your work position?
#Columns records are split by |

role_dummies = df["Which of the following best describes your work position?"].str.get_dummies(sep="|")

role_dummies = role_dummies.add_prefix("role_")

df = pd.concat([df, role_dummies], axis=1)

df = df.drop(columns=["Which of the following best describes your work position?"])


#Col 63 - Do you work remotely?
#Simple ordinal structure

order_column_63 = {"Never": 0, "Sometimes":1, "Always":2}
df["Do you work remotely?"] = df["Do you work remotely?"].map(order_column_63)



In [94]:
df

,Are you self-employed?,Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?,Do you have previous employers?,Have your previous employers provided mental health benefits?,Were you aware of the options for mental health care provided by your previous employers?,Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,Did your previous employers provide resources to learn more about mental health issues and how to seek help?,Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,Do you think that discussing a mental health disorder with previous employers would have negative consequences?,Do you think that discussing a physical health issue with previous employers would have negative consequences?,...,role_Dev Evangelist/Advocate,role_DevOps/SysAdmin,role_Executive Leadership,role_Front-end Developer,role_HR,role_One-person shop,role_Other,role_Sales,role_Supervisor/Team Lead,role_Support
0,0,0.0,1,0.0,0.0,NaN,0.0,NaN,1.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,0,0.0,1,2.0,2.0,0.0,1.0,2.0,0.0,0.0,...,0,0,0,1,0,0,0,0,0,0
2,0,0.0,1,0.0,0.0,0.0,1.0,NaN,NaN,1.0,...,0,0,0,0,0,0,0,0,0,0
3,1,NaN,1,1.0,0.0,0.0,0.0,NaN,1.0,1.0,...,0,0,0,0,0,0,0,0,1,0
4,0,0.0,1,NaN,0.0,1.0,0.0,NaN,1.0,1.0,...,1,1,1,1,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1428,1,NaN,1,2.0,2.0,1.0,1.0,NaN,NaN,0.0,...,0,0,0,0,0,0,1,0,0,0
1429,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,1
1430,0,1.0,1,1.0,2.0,0.0,1.0,1.0,2.0,1.0,...,0,0,0,0,0,0,0,0,0,0
1431,0,0.0,1,0.0,0.0,0.0,0.0,NaN,2.0,0.0,...,0,1,0,0,0,0,0,0,0,0


In [95]:
#Dealing with NA values - assigning neutral  values to missing values
explicit_fills = {
    "Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?": 0,
    "If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?": 0,
    "If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?": 0,
    "medical_leave_ordinal": 3,
    "discussing_mental_employer_ordinal": 2,
    "discussing_physical_employer_ordinal": 2,
    "discussing_mental_coworkers_ordinal": 2,
    "discussing_mental_supervisor_ordinal": 2,
}
for col, val in explicit_fills.items():
    df[col] = df[col].fillna(val)

In [96]:
#removing NA column produced by creation of dummy variables and columns  that have only one value
redundant_nan_cols = [
    "is_tech_comp?_nan", "company_size_nan", "healthcare_benefits_nan",
    "healthcare_anonymity_nan", "healthcare_employer_resources_nan",
    "healthcare_employer_discussion_nan", "employer_take_health_nan",
    "healthcare_awareness_nan", 'condition_Traumatic Brain Injury','continent_nan']
df = df.drop(columns=redundant_nan_cols)

In [97]:
#Remaining NA values, where this is not possible are replaced by median.

remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0].index.tolist()
for col in remaining_missing:
    df[col] = df[col].fillna(df[col].median())

In [98]:
#Check, if there are any NA values left. If there are = no PCA reduction.
print(df.isnull().sum().sum())

0


In [99]:
#saving the dataset

df.to_csv(r"C:\Users\ugniu\PycharmProjects\Unsupervised  learning\Data\Processed\mental_health_cleaned.csv", index=False)



In [100]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1428 entries, 0 to 1432
Columns: 106 entries, Are you self-employed? to role_Support
dtypes: bool(44), float64(22), int64(40)
memory usage: 764.2 KB


In [101]:
for col in df.columns:
    print(col)

Are you self-employed?
Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?
Do you have previous employers?
Have your previous employers provided mental health benefits?
Were you aware of the options for mental health care provided by your previous employers?
Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?
Did your previous employers provide resources to learn more about mental health issues and how to seek help?
Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?
Do you think that discussing a mental health disorder with previous employers would have negative consequences?
Do you think that discussing a physical health issue with previous employers would have negative consequences?
Would you have been willing to discuss a mental health iss